### `Importing Necessary`

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

### `Initializing Our Data Paths`

In [2]:
base = Path("../data/AGB")
csv_path = base / "data" / "Tapajos_inventory_data_2010.csv"
shp_path = base / "data" / "forest_inventory_tapajos" / "forest_inventory_tapajos.shp"

**Create geojson files for each plot**

### `Loading Our Data`

In [3]:
df = pd.read_csv(csv_path)
df.head().T

,0,1,2,3,4
plot,1,1,1,1,1
type,SF,SF,SF,SF,SF
origin,NE,NE,NE,NE,NE
tree,1,2,3,4,5
family,Hypericaceae,Salicaceae,Salicaceae,Salicaceae,Chrysobalanaceae
genus,Vismia,Banara,Banara,Banara,Couepia
scientific_name,Vismia guianensis,Banara nitida,Banara nitida,Banara nitida,Couepia sp.
common_name,Lacre branco,Cabelo de cutia,Cabelo de cutia,Cabelo de cutia,Macucu
density_wood,0.48,0.6,0.6,0.6,0.791
dbh,5.9,7.4,6.3,5.4,8.1


| S.N | Column Name                        | Description                                                                                  | Category                     |
|-----|------------------------------------|----------------------------------------------------------------------------------------------|------------------------------|
| 1   | plot                               | Plot ID linking each tree to a field plot                                                    | Identification               |
| 2   | tree                               | Tree ID within the plot                                                                      | Identification               |
| 3   | type                               | Forest type classification (e.g., SF = Secondary Forest)                                     | Stand attributes             |
| 4   | origin                             | Tree origin (e.g., natural, planted)                                                         | Stand attributes             |
| 5   | family                             | Botanical family                                                                              | Taxonomy                     |
| 6   | genus                              | Genus of the species                                                                         | Taxonomy                     |
| 7   | scientific_name                    | Scientific species name                                                                       | Taxonomy                     |
| 8   | common_name                        | Local or common tree name                                                                     | Taxonomy                     |
| 9   | density_wood                       | Wood density (g/cm³), critical for biomass calculations                                      | Wood property                |
| 10  | dbh                                | Diameter at breast height (cm), primary predictor for biomass                                | Tree measurement             |
| 11  | ht_total                           | Total height of the tree (m)                                                                  | Tree measurement             |
| 12  | ht_crown_base                      | Height where the crown begins (m)                                                             | Crown structure              |
| 13  | depth_crown                        | Vertical crown depth (ht_total – ht_crown_base)                                              | Crown structure              |
| 14  | stem_x                             | X-coordinate of tree stem inside the plot                                                     | Spatial position             |
| 15  | stem_y                             | Y-coordinate of tree stem inside the plot                                                     | Spatial position             |
| 16  | crown_radius_x0                    | Crown radius along X-axis at 0% crown height                                                  | Crown radius                 |
| 17  | crown_radius_x50                   | Crown radius along X-axis at 50% crown height                                                 | Crown radius                 |
| 18  | crown_radius_y0                    | Crown radius along Y-axis at 0% crown height                                                  | Crown radius                 |
| 19  | crown_radius_y50                   | Crown radius along Y-axis at 50% crown height                                                 | Crown radius                 |
| 20  | ht_crown_max_x0                    | Maximum crown height along X-axis at 0% crown height                                          | Crown height profile         |
| 21  | ht_crown_max_x50                   | Maximum crown height along X-axis at 50% crown height                                         | Crown height profile         |
| 22  | ht_crown_max_y0                    | Maximum crown height along Y-axis at 0% crown height                                          | Crown height profile         |
| 23  | ht_crown_max_y50                   | Maximum crown height along Y-axis at 50% crown height                                         | Crown height profile         |
| 24  | shape_coeff_crown_x0               | Crown shape coefficient along X-axis (lower section)                                          | Crown shape                  |
| 25  | shape_coeff_crown_x50              | Crown shape coefficient along X-axis (upper section)                                          | Crown shape                  |
| 26  | shape_coeff_crown_y0               | Crown shape coefficient along Y-axis (lower section)                                          | Crown shape                  |
| 27  | shape_coeff_crown_y50              | Crown shape coefficient along Y-axis (upper section)                                          | Crown shape                  |
| 28  | shape_coeff_crown_below_x0         | Crown shape coefficient below crown base along X-axis                                         | Crown shape (lower crown)    |
| 29  | shape_coeff_crown_below_x50        | Crown shape coefficient below crown base along X-axis (mid-level)                             | Crown shape (lower crown)    |
| 30  | shape_coeff_crown_below_y0         | Crown shape coefficient below crown base along Y-axis                                         | Crown shape (lower crown)    |
| 31  | shape_coeff_crown_below_y50        | Crown shape coefficient below crown base along Y-axis (mid-level)                             | Crown shape (lower crown)    |

In [4]:
gdf = gpd.read_file(shp_path)
gdf.head()

,plot,elev_m,area_ha,geometry
0,6,162.5,0.25,"POLYGON Z ((725874.269 9665420.64 0, 725866.96..."
1,5,138.9,0.25,"POLYGON Z ((725949.641 9664904.35 0, 725942.42..."
2,4,144.9,0.25,"POLYGON Z ((725924.612 9665076.496 0, 725917.3..."
3,3,106.2,0.25,"POLYGON Z ((727510.117 9653360.183 0, 727503.1..."
4,2,102.4,0.25,"POLYGON Z ((727500.82 9653453.015 0, 727491.45..."


| S.N | Column Name | Description                                                          | Category              |
| --- | ----------- | -------------------------------------------------------------------- | --------------------- |
| 32  | geometry    | Polygon geometry of the plot (coordinates in CRS, usually EPSG:xxxx) | Spatial/Plot boundary |
| 33  | elev_m      | Average elevation of the plot in meters                              | Plot attribute        |
| 34  | area_ha     | Area of the plot in hectares                                         | Plot attribute        |

### `Calculate AGB using Allometric Equation`

$$
\text{AGB} = 0.0673 \times (\rho \cdot \text{DBH}^2 \cdot H)^{0.976}
$$

Where:

- **AGB** = Aboveground biomass (kg or Mg, depending on units)
- **ρ (rho)** = Wood density (g/cm³)
- **DBH** = Diameter at breast height (cm)
- **H** = Total tree height (m)

> This equation is commonly used for tropical trees.

In [5]:
df_agb = df[['plot', 'density_wood', 'dbh', 'ht_total']].copy()
df_agb.head()

,plot,density_wood,dbh,ht_total
0,1,0.480,5.9,6.0
1,1,0.600,7.4,7.1
2,1,0.600,6.3,7.1
3,1,0.600,5.4,4.3
4,1,0.791,8.1,9.3


In [6]:
# Calculate AGB per tree (kg)
df_agb["agb_kg"] = 0.0673* (df_agb["density_wood"] * (df_agb["dbh"] ** 2) * df_agb["ht_total"]) ** 0.976
agb_plot_kg = df_agb.groupby("plot")["agb_kg"].sum().reset_index()

In [7]:
# shapefile with 'plot' and 'area_ha'
agb_plot = pd.merge(agb_plot_kg, gdf[["plot", "area_ha"]], on="plot", how="left")

## Convert to tonnes per hectare (t/ha)
# 1 tonne = 1000 kg
# Divide by plot area in ha
agb_plot["agb_t_ha"] = (agb_plot["agb_kg"] / 1000) / agb_plot["area_ha"]
agb_plot[["plot", "agb_t_ha"]]

,plot,agb_t_ha
0,1,48.085583
1,2,131.077836
2,3,125.887898
3,4,229.134275
4,5,220.324153
5,6,299.540070
6,7,380.895435
7,8,314.055965
8,9,312.320233
9,10,420.295380
